In [1]:
import os
import urllib.request

# 1. Clone the repo to get their scripts and structure
!git clone https://github.com/qcri/DisasterVQA.git

# 2. Check the dataset path directly inside the repo
dataset_path = "DisasterVQA/dataset/disasterVQA_dataset.json"

Cloning into 'DisasterVQA'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 128 (delta 60), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 4.15 MiB | 12.95 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [2]:
import json

dataset_path = "DisasterVQA/dataset/disasterVQA_dataset.json"

with open(dataset_path, "r") as f:
    vqa_data = json.load(f)

print(f"Total QA pairs loaded: {len(vqa_data)}")
print("\nSample Data Entry:")
print(json.dumps(vqa_data[0], indent=2))

Total QA pairs loaded: 4405

Sample Data Entry:
{
  "id": "imgq-1",
  "image_id": "img-1",
  "image_path": "DisasterVQA/MEDIC/bridge_damage_1600.jpg",
  "region": null,
  "disaster_type": "landslide",
  "dataset_source": "medic",
  "question_type": "Yes/No",
  "question": "Is there visible damage to infrastructure in the image? Answer with Yes or No.",
  "groundtruth_answer": "Yes",
  "crisis_info_type": "situational_awareness",
  "crisis_info_code": "SA-2"
}


In [ ]:
import json
import pandas as pd

# Load DisasterVQA raw dataset using the correct path from git clone
dataset_path = "DisasterVQA/dataset/disasterVQA_dataset.json"
with open(dataset_path, "r") as f:
    vqa_data = json.load(f)

formatted_samples = []

for item in vqa_data:
    image_path = item.get("image_path")
    question = item.get("question")
    answer = item.get("answer")
    q_type = item.get("question_type") # binary, mcq, or open-ended
    category = item.get("humanitarian_category", "General")

    # Structure instruction based on question type
    if q_type == "mcq":
        choices = "\n".join([f"{k}: {v}" for k, v in item.get("choices", {}).items()])
        instruction = f"{question}\nChoices:\n{choices}\nAnswer with option letter and brief explanation."
    else:
        instruction = f"{question}\nProvide a concise and accurate answer."

    formatted_samples.append({
        "id": item.get("id"),
        "image": image_path,
        "instruction": instruction,
        "ground_truth": answer,
        "question_type": q_type,
        "humanitarian_category": category
    })

# Save to jsonl for evaluation
with open("disaster_vqa_eval.jsonl", "w") as f:
    for entry in formatted_samples:
        f.write(json.dumps(entry) + "\n")

print(f"Successfully processed {len(formatted_samples)} DisasterVQA evaluation triplets!")

Successfully processed 4405 DisasterVQA evaluation triplets!


In [4]:
import os
from kaggle_secrets import UserSecretsClient

# Fetch HF_TOKEN from Kaggle Secrets
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [5]:
import json
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# 1. Load Model & Processor
model_id = "Qwen/Qwen2-VL-7B-Instruct"  # Or load your local/merged QLoRA model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

results = []

# 2. Batch Inference Loop
with open("disaster_vqa_eval.jsonl", "r") as f:
    lines = [json.loads(line) for line in f]

# For quick test/eval, you can test on first 100-200 samples if short on time
for sample in lines[:200]:  
    img_path = sample["image"]
    
    # Handle missing/local images gracefully
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        continue

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": sample["instruction"]},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    sample["model_output"] = output_text
    results.append(sample)

# Save predictions
with open("vqa_predictions.jsonl", "w") as f:
    for item in results:
        f.write(json.dumps(item) + "\n")

print("Inference completed! Predictions saved to vqa_predictions.jsonl")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Inference completed! Predictions saved to vqa_predictions.jsonl


In [6]:
import json
import os
import time
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# Fetch API key securely from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")

# Initialize Gemini Client with the fetched key
client = genai.Client(api_key=api_key)

def evaluate_open_ended(question, ground_truth, model_output):
    """Uses Gemini 2.5 Flash as a judge to evaluate open-ended semantic equality."""
    prompt = f"""
    You are an expert disaster response evaluator.
    Question: {question} 
    Ground Truth Answer: {ground_truth} 
    Model Predicted Answer: {model_output} 
    Is the Model Predicted Answer semantically equivalent and factually consistent with the Ground Truth Answer?
    Respond ONLY with a JSON object: {{"verdict": "CORRECT"}} or {{"verdict": "INCORRECT"}}
    """
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        res_json = json.loads(response.text)
        return 1 if res_json.get("verdict") == "CORRECT" else 0
    except Exception as e:
        time.sleep(1)  # Rate limit backing off
        return 0

# Metrics computation
binary_correct = 0
binary_total = 0
open_correct = 0
open_total = 0

with open("vqa_predictions.jsonl", "r") as f:
    predictions = [json.loads(line) for line in f]

for pred in predictions:
    q_type = pred["question_type"]
    gt = str(pred["ground_truth"]).strip().lower()
    out = str(pred["model_output"]).strip().lower()
    
    if q_type == "binary":
        binary_total += 1
        if ("yes" in gt and "yes" in out) or ("no" in gt and "no" in out):
            binary_correct += 1
    elif q_type == "open-ended":
        open_total += 1
        score = evaluate_open_ended(pred["instruction"], pred["ground_truth"], pred["model_output"])
        open_correct += score

binary_accuracy = (binary_correct / binary_total) * 100 if binary_total > 0 else 0
open_accuracy = (open_correct / open_total) * 100 if open_total > 0 else 0

print("--- DisasterVQA Benchmark Results ---")
print(f"Binary Question Accuracy: {binary_accuracy:.2f}% ({binary_correct}/{binary_total})")
print(f"Open-Ended QA Accuracy (LLM-Judge): {open_accuracy:.2f}% ({open_correct}/{open_total})")

--- DisasterVQA Benchmark Results ---
Binary Question Accuracy: 0.00% (0/0)
Open-Ended QA Accuracy (LLM-Judge): 0.00% (0/0)


In [7]:
import json
import os
import time
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# 1. Fetch API key securely from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def evaluate_open_ended(question, ground_truth, model_output):
    """Uses Gemini 2.5 Flash as a judge to evaluate open-ended semantic equality."""
    prompt = f"""
    You are an expert disaster response evaluator.
    Question: {question} 
    Ground Truth Answer: {ground_truth} 
    Model Predicted Answer: {model_output} 
    Is the Model Predicted Answer semantically equivalent and factually consistent with the Ground Truth Answer?
    Respond ONLY with a JSON object: {{"verdict": "CORRECT"}} or {{"verdict": "INCORRECT"}}
    """
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(response_mime_type="application/json")
        )
        res_json = json.loads(response.text)
        return 1 if res_json.get("verdict") == "CORRECT" else 0
    except Exception as e:
        time.sleep(1)
        return 0

# 2. Metrics computation
binary_correct = 0
binary_total = 0
open_correct = 0
open_total = 0

with open("vqa_predictions.jsonl", "r") as f:
    predictions = [json.loads(line) for line in f]

for pred in predictions:
    q_type = pred["question_type"]
    gt = str(pred["ground_truth"]).strip().lower()
    out = str(pred["model_output"]).strip().lower()
    
    if q_type == "binary":
        binary_total += 1
        if ("yes" in gt and "yes" in out) or ("no" in gt and "no" in out):
            binary_correct += 1
    elif q_type == "open-ended":
        open_total += 1
        score = evaluate_open_ended(pred["instruction"], pred["ground_truth"], pred["model_output"])
        open_correct += score

binary_accuracy = (binary_correct / binary_total) * 100 if binary_total > 0 else 0
open_accuracy = (open_correct / open_total) * 100 if open_total > 0 else 0

print("--- DisasterVQA Benchmark Results ---")
print(f"Binary Question Accuracy: {binary_accuracy:.2f}% ({binary_correct}/{binary_total})")
print(f"Open-Ended QA Accuracy (LLM-Judge): {open_accuracy:.2f}% ({open_correct}/{open_total})")

--- DisasterVQA Benchmark Results ---
Binary Question Accuracy: 0.00% (0/0)
Open-Ended QA Accuracy (LLM-Judge): 0.00% (0/0)


In [3]:
import json
import torch
import sys
import types
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

# 1. APPLY THE TORCHCODEC PATCH TO PREVENT KAGGLE CRASHES
try:
    import torchcodec
except Exception:
    dummy_tc = types.ModuleType("torchcodec")
    dummy_dec = types.ModuleType("torchcodec.decoders")
    class DummyVideoDecoder: pass
    dummy_dec.VideoDecoder = DummyVideoDecoder
    dummy_tc.decoders = dummy_dec
    sys.modules["torchcodec"] = dummy_tc
    sys.modules["torchcodec.decoders"] = dummy_dec
    print("✅ torchcodec patch applied successfully.")

# 2. LOAD YOUR TEAM'S FINE-TUNED MODEL
model_id = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"
print(f"Loading model: {model_id} ... (This may take a few minutes)")

# Use the Qwen2_5 specific class
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# 3. PREPARE THE DATA 
input_file = "disaster_vqa_eval.jsonl"
output_file = "vqa_real_predictions.jsonl"
results = []

print(f"Loading evaluation data from {input_file}...")
with open(input_file, "r") as f:
    samples = [json.loads(line) for line in f]

# 4. RUN INFERENCE LOOP
print("Starting inference on DisasterVQA images...")

for i, sample in enumerate(samples):
    img_path = sample["image"]
    
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"⚠️ Skipping image {img_path}: {e}")
        continue

    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": sample["instruction"]},
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    sample["model_output"] = output_text
    results.append(sample)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} / {len(samples)} images...")

# 5. SAVE THE FINAL FILE
with open(output_file, "w") as f:
    for item in results:
        f.write(json.dumps(item) + "\n")

print(f"\n🎉 SUCCESS! All predictions have been saved to '{output_file}'.")


Loading model: AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP ... (This may take a few minutes)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loading evaluation data from disaster_vqa_eval.jsonl...


FileNotFoundError: [Errno 2] No such file or directory: 'disaster_vqa_eval.jsonl'

In [ ]:
import os
print("File size:", os.path.getsize("vqa_real_predictions.jsonl"), "bytes")

File size: 0 bytes


In [ ]:
import json
import pandas as pd
import os
import time
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# 1. Initialize Gemini Client using Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
except Exception as e:
    raise ValueError(
        "Could not find GEMINI_API_KEY. Please ensure it is added and toggled"
        " ON in Kaggle Secrets."
    )

client = genai.Client(api_key=api_key)

def evaluate_open_ended(question, ground_truth, model_output):
    """Uses Gemini 2.5 Flash as a judge to evaluate open-ended semantic equality."""
    prompt = f"""
    You are an expert disaster response evaluator.
    Question: {question}
    Ground Truth Answer: {ground_truth}
    Model Predicted Answer: {model_output}
    Is the Model Predicted Answer semantically equivalent and factually consistent with the Ground Truth Answer?
    Respond ONLY with a JSON object: {{"verdict": "CORRECT"}} or {{"verdict": "INCORRECT"}}
    """
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            ),
        )
        res_json = json.loads(response.text)
        return 1 if res_json.get("verdict") == "CORRECT" else 0
    except Exception as e:
        time.sleep(2)
        return 0

print("Loading raw predictions from vqa_predictions.jsonl...")
with open("vqa_real_predictions.jsonl", "r") as f:
    predictions = [json.loads(line) for line in f]

print("Evaluating answers (this may take a moment for the Gemini API)...")
for pred in predictions:
    q_type = pred.get("question_type", "")
    gt = str(pred.get("ground_truth", "")).strip().lower()
    out = str(pred.get("model_output", "")).strip().lower()
    
    if q_type == "binary":
        if ("yes" in gt and "yes" in out) or ("no" in gt and "no" in out):
            pred["is_correct"] = 1
        else:
            pred["is_correct"] = 0
    else:
        pred["is_correct"] = evaluate_open_ended(pred.get("instruction", ""), gt, out)

# 2. SAVE THE MISSING FILE 
with open("vqa_real_predictions_evaluated.jsonl", "w") as f:
    for pred in predictions:
        f.write(json.dumps(pred) + "\n")
print("✓ Saved evaluated predictions to 'vqa_real_predictions_evaluated.jsonl'")

# 3. GENERATE THE CATEGORY BREAKDOWN TABLE
df = pd.DataFrame(predictions)

# Ensure fallback values exist for both grouping columns to prevent KeyErrors
if 'humanitarian_category' not in df.columns:
    df['humanitarian_category'] = 'General'
else:
    df['humanitarian_category'] = df['humanitarian_category'].fillna('General')

if 'question_type' not in df.columns:
    df['question_type'] = 'Unknown'
else:
    df['question_type'] = df['question_type'].fillna('Unknown')

# Safe aggregation
summary = (
    df.groupby(['humanitarian_category', 'question_type'])['is_correct']
    .agg(
        Total='count',
        Correct='sum',
        Accuracy=lambda x: (x.sum() / x.count()) * 100 if x.count() > 0 else 0
    )
    .reset_index()
)

print("\n=== DisasterVQA Category Breakdown ===")
print(summary.to_string(index=False))

# Export to CSV for your project report
summary.to_csv("disastervqa_category_metrics.csv", index=False)
print("\n✓ Successfully saved 'disastervqa_category_metrics.csv'!")

Loading raw predictions from vqa_predictions.jsonl...
Evaluating answers (this may take a moment for the Gemini API)...
✓ Saved evaluated predictions to 'vqa_predictions_evaluated.jsonl'


KeyError: 'Column not found: is_correct'